# Aktivitas Naive Bayes — Apakah Cocok Bermain Tenis Hari Ini?

**Mata Kuliah:** DSB07 — Machine Learning  
**Topik:** Naive Bayes (Pertemuan 10)  
**Durasi:** 45 menit  

## 👥 Anggota Kelompok

| No | Nama | NIM |
|---:|------|-----|
| 1  | Joel Prasetya Pradipta     | 32230020     |
| 2  | Richard Stefano     | 32230002    |
| 3  |   Richard Sidharta   |  32230069   |
| 4  |  Yogi Tansyah    |  32220162   |
| 5  |  Ruben Wijaya    |  32230048   |
| 6  |  Harlan Luthi Permana    |  32230033   |
| 7  | Cristian Iverson Jong    |  32230057   |

## Petunjuk
Lengkapi setiap sel berlabel `# TODO`. Jangan ubah sel yang sudah berisi kode lengkap kecuali diminta.
Jalankan sel secara berurutan dari atas ke bawah.

## Setup

In [4]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

pd.set_option("display.precision", 4)
np.set_printoptions(precision=4, suppress=True)

## Tahap 1 — Memahami Data (5 menit)

Dataset `play_tennis` berisi 14 hari pengamatan apakah cocok untuk bermain tenis berdasarkan kondisi cuaca.

| Fitur | Nilai mungkin |
|-------|---------------|
| Outlook | Sunny, Overcast, Rain |
| Temperature | Hot, Mild, Cool |
| Humidity | High, Normal |
| Wind | Weak, Strong |
| **Play** (label) | **Yes, No** |

In [5]:
# Ganti USERNAME dengan akun GitHub dosen Anda setelah repo di-push.
url = "https://raw.githubusercontent.com/UBM-ML/naive-bayes/main/data/play_tennis.csv"
df = pd.read_csv(url)
df

,Outlook,Temperature,Humidity,Wind,Play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [6]:
# TODO 1: Tampilkan distribusi label 'Play' (berapa Yes, berapa No)
# Hint: gunakan .value_counts() pada kolom 'Play'
...

Ellipsis

## Tahap 2 — Hitung Manual (15 menit)

**Kasus uji:** Hari ini cuaca **Sunny**, suhu **Cool**, kelembapan **High**, angin **Strong**.  
**Pertanyaan:** Apakah cocok bermain tenis (`Play = Yes` atau `No`)?

Rumus (slide 2.2):
$$P(K \mid F_1, F_2, \dots, F_n) \propto P(K) \times \prod_{i=1}^{n} P(F_i \mid K)$$

Kita akan menghitung *tanpa* Laplace smoothing terlebih dahulu (sesuai rumus di slide).

In [7]:
test = {"Outlook": "Sunny", "Temperature": "Cool", "Humidity": "High", "Wind": "Strong"}
test

{'Outlook': 'Sunny',
 'Temperature': 'Cool',
 'Humidity': 'High',
 'Wind': 'Strong'}

### 2.1 Probabilitas Prior P(K)

In [16]:
# TODO 2: Hitung P(Yes) dan P(No)
n_total = len(df)   # total baris
n_yes = (df['Play'] == 'Yes').sum()     # jumlah Play == 'Yes'
n_no = (df['Play'] == 'No').sum()      # jumlah Play == 'No'

p_yes = n_yes / n_total
p_no = n_no / n_total

print(f"P(Yes) = {p_yes:.4f}")
print(f"P(No)  = {p_no:.4f}")

P(Yes) = 0.6429
P(No)  = 0.3571


### 2.2 Probabilitas Kondisional P(F_i | K)

Untuk setiap fitur dalam kasus uji, hitung berapa kali nilainya muncul dalam tiap kelas, dibagi total baris kelas tersebut.

In [11]:
df_yes = df[df["Play"] == "Yes"]
df_no = df[df["Play"] == "No"]

# TODO 3: Hitung 4 probabilitas kondisional untuk kelas Yes
# Contoh: p_sunny_yes = (df_yes['Outlook'] == 'Sunny').sum() / len(df_yes)
p_sunny_yes  = (df_yes['Outlook'] == 'Sunny').sum() / len(df_yes)
p_cool_yes   = (df_yes['Temperature'] == 'Cool').sum() / len(df_yes)
p_high_yes   = (df_yes['Humidity'] == 'High').sum() / len(df_yes)
p_strong_yes = (df_yes['Wind'] == 'Strong').sum() / len(df_yes)

# TODO 4: Hitung 4 probabilitas kondisional untuk kelas No
p_sunny_no  = (df_no['Outlook'] == 'Sunny').sum() / len(df_no)
p_cool_no   = (df_no['Temperature'] == 'Cool').sum() / len(df_no)
p_high_no   = (df_no['Humidity'] == 'High').sum() / len(df_no)
p_strong_no = (df_no['Wind'] == 'Strong').sum() / len(df_no)

print("Kelas Yes:", p_sunny_yes, p_cool_yes, p_high_yes, p_strong_yes)
print("Kelas No :", p_sunny_no,  p_cool_no,  p_high_no,  p_strong_no)

Kelas Yes: 0.2222222222222222 0.3333333333333333 0.3333333333333333 0.3333333333333333
Kelas No : 0.6 0.2 0.8 0.6


### 2.3 Posterior & Keputusan

In [20]:
# TODO 5: Hitung skor posterior (proporsional, belum dinormalisasi)
score_yes = p_yes * p_sunny_yes * p_cool_yes * p_high_yes * p_strong_yes
score_no  = p_no  * p_sunny_no  * p_cool_no  * p_high_no  * p_strong_no

# TODO 6: Normalisasi sehingga total = 1
total = score_yes + score_no
post_yes = score_yes / total
post_no  = score_no / total

prediksi_manual = "Yes" if post_yes > post_no else "No"

print(f"P(Yes | x) = {post_yes:.4f}")
print(f"P(No  | x) = {post_no:.4f}")
print(f"Prediksi manual: {prediksi_manual}")

P(Yes | x) = 0.2046
P(No  | x) = 0.7954
Prediksi manual: No


## Tahap 3 — Implementasi dengan scikit-learn (15 menit)

`CategoricalNB` di sklearn membutuhkan input numerik, jadi fitur kategorikal kita ubah dulu dengan `OrdinalEncoder`.

In [18]:
features = ["Outlook", "Temperature", "Humidity", "Wind"]

encoder = OrdinalEncoder()
X = encoder.fit_transform(df[features])
y = df["Play"].values

print("Pemetaan kategori -> angka:")
for col, cats in zip(features, encoder.categories_):
    print(f"  {col}: {dict(enumerate(cats))}")

Pemetaan kategori -> angka:
  Outlook: {0: 'Overcast', 1: 'Rain', 2: 'Sunny'}
  Temperature: {0: 'Cool', 1: 'Hot', 2: 'Mild'}
  Humidity: {0: 'High', 1: 'Normal'}
  Wind: {0: 'Strong', 1: 'Weak'}


In [35]:
# TODO 7: Latih CategoricalNB (alpha default = 1, pakai Laplace smoothing)
model = CategoricalNB(alpha=1)
model.fit(X, y)

modelTanpaSmoothing = CategoricalNB(alpha=1e-10)
modelTanpaSmoothing.fit(X, y)

# TODO 8: Hitung akurasi pada data latih (perlu .predict(X) dulu)
y_pred = model.predict(X)
akurasi = accuracy_score(y, y_pred)
print(f"Akurasi pada data latih: {akurasi:.4f}")

y_pred = modelTanpaSmoothing.predict(X)
akurasi = accuracy_score(y, y_pred)
print(f"Akurasi pada data latih: {akurasi:.4f}")

Akurasi pada data latih: 0.9286
Akurasi pada data latih: 0.9286


In [43]:
# TODO 9: Prediksi kasus uji yang SAMA dengan Tahap 2
x_test_df = pd.DataFrame([test])
x_test = encoder.transform(x_test_df[features])

prediksi_sklearn = model.predict(x_test)   # gunakan model.predict
proba_sklearn = model.predict_proba(x_test)      # gunakan model.predict_proba
print('Model dengan Alpha = 1')
print(f"Prediksi sklearn : {prediksi_sklearn[0]}")
print(f"Kelas-kelas      : {model.classes_}")
print(f"Probabilitas     : {proba_sklearn[0]}")

prediksi_sklearn = modelTanpaSmoothing.predict(x_test)   # gunakan model.predict
proba_sklearn = modelTanpaSmoothing.predict_proba(x_test)      # gunakan model.predict_proba
print('\n')
print('Model dengan Alpha = 1e-10')
print(f"Prediksi sklearn : {prediksi_sklearn[0]}")
print(f"Kelas-kelas      : {model.classes_}")
print(f"Probabilitas     : {proba_sklearn[0]}")

Model dengan Alpha = 1
Prediksi sklearn : No
Kelas-kelas      : ['No' 'Yes']
Probabilitas     : [0.7201 0.2799]


Model dengan Alpha = 1e-10
Prediksi sklearn : No
Kelas-kelas      : ['No' 'Yes']
Probabilitas     : [0.7954 0.2046]


## Tahap 4 — Diskusi Kelompok (7 menit)

_Tulis jawaban kelompok dalam sel di bawah._

1. Apakah **prediksi kelas** manual sama dengan sklearn? Apakah **nilai probabilitas**-nya juga sama? Jika berbeda, kira-kira kenapa?  
   _Hint: cari arti parameter `alpha` di `CategoricalNB` (Laplace / additive smoothing)._
2. Coba ubah `CategoricalNB(alpha=1e-10)` dan jalankan ulang Tahap 3. Apakah probabilitas sklearn jadi lebih dekat ke hasil manual?
3. Apa yang akan terjadi jika kasus uji mengandung kombinasi (kategori, kelas) yang **tidak pernah muncul** di data latih (misalnya `Outlook=Overcast` pada kelas `No`)? Bagaimana Laplace smoothing menyelamatkan situasi ini?  
   _Hint: lihat slide 1.3 poin #3 (Sensitif terhadap Atribut yang Hilang)._
4. Naive Bayes mengasumsikan **semua fitur saling independen**. Menurut kelompok Anda, apakah `Humidity` dan `Outlook` benar-benar independen di dunia nyata? Apa konsekuensinya untuk akurasi model?

**Jawaban Kelompok:**

1. **Prediksi vs Probabilitas:** Prediksi kelasnya **sama** (keduanya memprediksi 'No'). Namun, nilai probabilitasnya **berbeda** (Manual: ~0.7954 vs Sklearn: 0.7201). Hal ini terjadi karena scikit-learn secara default menggunakan `alpha=1` (Laplace Smoothing), sedangkan perhitungan manual kita di Tahap 2 tidak menggunakannya.
2. **Eksperimen Alpha:** Ya, jika `alpha` diubah menjadi nilai yang sangat kecil (seperti `1e-10`), efek smoothing menghilang dan nilai probabilitas scikit-learn akan menjadi sangat mendekati (identik) dengan hasil hitung manual.
3. **Kategori Hilang:** Jika ada kombinasi yang tidak pernah muncul (frekuensi 0), probabilitas kondisionalnya akan menjadi 0. Tanpa Laplace smoothing, skor akhir (posterior) akan menjadi 0 karena dikalikan. Laplace smoothing menambahkan nilai kecil ke pembilang sehingga probabilitas tidak pernah nol, 'menyelamatkan' model dari kegagalan prediksi pada data baru.
4. **Independensi:** Di dunia nyata, `Humidity` dan `Outlook` seringkali **tidak independen** (misalnya, saat 'Rain', kelembapan cenderung 'High'). Konsekuensi dari asumsi Naive Bayes yang melanggar kenyataan ini adalah model mungkin menjadi terlalu 'percaya diri' (overconfident) pada probabilitasnya, meskipun seringkali akurasi klasifikasinya tetap cukup baik.

## Tahap 5 — Refleksi (3 menit)

Tulis 3 kalimat singkat:
- Hal terpenting yang saya pelajari hari ini adalah cara kerja algoritma Naive Bayes dan bagaimana parameter `alpha` (Laplace Smoothing) mempengaruhi hasil probabilitas akhir.
- Bagian paling sulit adalah memastikan perhitungan probabilitas kondisional manual dilakukan dengan teliti sesuai dengan label kelasnya.
- Naive Bayes cocok dipakai ketika kita memiliki dataset dengan fitur kategorikal yang independen dan membutuhkan model yang cepat serta efisien.

## Submission
Simpan notebook ini ke Google Drive masing-masing, lalu kumpulkan link **share** (akses *Anyone with the link → Viewer*) ke kanal kelas yang ditentukan dosen.